# Statistical Arbitrage: Pairs Trading with Cointegration**Market-Neutral Statistical Arbitrage using ARBS Framework**This notebook demonstrates **cointegration-based pairs trading** integrated with the ARBS backtesting framework.## What You'll Learn1. **ARBS Integration**: Implementing PairsSignal extending BaseSignal2. **Cointegration Testing**: Engle-Granger methodology for pair selection3. **Signal Generation**: Z-score based mean-reversion signals4. **Portfolio Construction**: Using MeanVarianceOptimizer with pairs signals5. **Full Pipeline**: BaseSignal pattern for custom signals6. **Performance Analysis**: Market-neutral returns with comprehensive tear sheet## Key Concepts**Cointegration**: Two non-stationary price series that have a stationary linear combination (spread)**Pairs Trading**: Long undervalued asset, short overvalued asset, profit when spread mean-reverts**Market-Neutral**: Dollar-neutral positions eliminate market exposure (beta ≈ 0)**Paper References**:- Engle & Granger (1987): "Co-integration and Error Correction"- Gatev, Goetzmann & Rouwenhorst (2006): "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"- Do & Faff (2010): "Does Simple Pairs Trading Still Work?"

---

## Setup

In [ ]:
# Add parent directory to pathimport sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))# Standard importsimport numpy as npimport pandas as pdimport polars as plfrom datetime import date, timedeltafrom typing import List, Tuple, Dict, Optional# ARBS Framework Importsfrom Signals.Base.BaseSignal import BaseSignalfrom Signals.AlphaGenerator import AlphaGeneratorfrom Risk.Covariance.LedoitWolfShrinkage import LedoitWolfShrinkagefrom Optimizer.MeanVarianceOptimizer import MeanVarianceOptimizerfrom Risk.Returns.ReturnsCalculator import ReturnsCalculatorfrom Risk.Volatility.RealizedVolatility import RealizedVolatility# Statistical testsfrom statsmodels.tsa.stattools import cointfrom sklearn.linear_model import LinearRegression# Visualizationimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (14, 6)# Set random seed for reproducibilitynp.random.seed(42)print("✓ ARBS Framework loaded successfully!")

---

## Part 1: Generate Mock Stock Data

Create synthetic cointegrated stock data for testing.

In [ ]:
# Define stocks and sectors
stocks = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'GOOGL': 'Technology',
    'NVDA': 'Technology',
    'JPM': 'Financials',
    'BAC': 'Financials',
    'GS': 'Financials',
    'XOM': 'Energy',
    'CVX': 'Energy',
    'COP': 'Energy'
}

tickers = list(stocks.keys())
n_days = 504
dates = [date(2022, 1, 1) + timedelta(days=i) for i in range(n_days)]

def generate_cointegrated_prices(n_days: int, stocks_dict: Dict[str, str]) -> Dict[str, np.ndarray]:
    """
    Generate price series with cointegration within sectors.
    
    Strategy:
    1. Generate sector-level common factor (random walk)
    2. Each stock = sector_factor + idiosyncratic_factor
    3. Within sector, stocks share cointegration relationship
    """
    # Group stocks by sector
    sector_stocks = {}
    for ticker, sector in stocks_dict.items():
        if sector not in sector_stocks:
            sector_stocks[sector] = []
        sector_stocks[sector].append(ticker)
    
    prices = {}
    
    # Generate prices for each sector
    for sector, sector_tickers in sector_stocks.items():
        n_stocks = len(sector_tickers)
        
        # Generate common sector factor (non-stationary random walk)
        sector_shocks = np.random.normal(0, 0.015, n_days)
        sector_factor = 100 + np.cumsum(sector_shocks)
        
        # Generate mean-reverting component (stationary)
        phi = 0.95  # Mean reversion speed
        mean_reverting = np.zeros(n_days)
        for t in range(1, n_days):
            shock = np.random.normal(0, 0.5)
            mean_reverting[t] = phi * mean_reverting[t-1] + shock
        
        # Each stock = sector_factor + stock_specific + mean_reverting
        for i, ticker in enumerate(sector_tickers):
            drift = np.random.normal(0, 0.001, n_days).cumsum()
            idiosyncratic = np.random.normal(0, 0.005, n_days).cumsum()
            
            stock_price = (
                0.85 * sector_factor +
                0.15 * idiosyncratic +
                0.3 * mean_reverting +
                drift
            )
            
            stock_price = np.maximum(stock_price, 10)
            prices[ticker] = stock_price
    
    return prices

# Generate cointegrated prices
prices_dict = generate_cointegrated_prices(n_days, stocks)

# Create Polars DataFrame
prices_data = []
for ticker in tickers:
    for i, d in enumerate(dates):
        prices_data.append({
            'date': d,
            'ticker': ticker,
            'price': prices_dict[ticker][i],
            'sector': stocks[ticker]
        })

prices_df = pl.DataFrame(prices_data)

print(f"✓ Generated {n_days} days of cointegrated price data for {len(tickers)} stocks")

---

## Part 2: Implement PairsSignal (ARBS Integration)

**Key ARBS Pattern**: Extend BaseSignal to implement custom signal logic.

PairsSignal workflow:
1. Find cointegrated pairs using Engle-Granger test
2. Calculate spreads and z-scores for each pair
3. Generate mean-reversion signals
4. Return signal matrix compatible with AlphaGenerator

In [ ]:
class PairsSignal:    """    Cointegration-based pairs trading signal.        Note: This is a standalone signal class, not extending BaseSignal,    because pairs trading requires analyzing multiple assets simultaneously.        Workflow:    1. Test all possible pairs for cointegration (Engle-Granger)    2. For cointegrated pairs, calculate spread = price_A - β × price_B    3. Calculate z-score = (spread - rolling_mean) / rolling_std    4. Generate signals: z > +2.0 → short spread, z < -2.0 → long spread    5. Exit when z crosses 0        Args:        prices_df: Polars DataFrame with columns [date, ticker, price]        min_correlation: Minimum correlation for pair consideration        significance_level: P-value threshold for cointegration test        z_score_window: Rolling window for z-score calculation        entry_threshold: Z-score threshold for entry (default: 2.0)        exit_threshold: Z-score threshold for exit (default: 0.0)    """        def __init__(self,                 prices_df: pl.DataFrame,                 min_correlation: float = 0.70,                 significance_level: float = 0.05,                 z_score_window: int = 20,                 entry_threshold: float = 2.0,                 exit_threshold: float = 0.0):        self.prices_df = prices_df        self.min_correlation = min_correlation        self.significance_level = significance_level        self.z_score_window = z_score_window        self.entry_threshold = entry_threshold        self.exit_threshold = exit_threshold                # Store cointegrated pairs and their metadata        self.pairs = []        self.hedge_ratios = {}            def _test_cointegration(self, price_A: np.ndarray, price_B: np.ndarray) -> Tuple[bool, float]:        """Test if two price series are cointegrated."""        t_stat, p_value, _ = coint(price_A, price_B)                # Estimate hedge ratio via OLS        model = LinearRegression()        model.fit(price_B.reshape(-1, 1), price_A)        hedge_ratio = model.coef_[0]                cointegrated = p_value < self.significance_level                return cointegrated, hedge_ratio        def _find_cointegrated_pairs(self, prices_wide: pd.DataFrame) -> None:        """Find all cointegrated pairs."""        tickers = prices_wide.columns.tolist()                # Calculate correlation matrix        corr_matrix = prices_wide.corr()                # Test all pairs with sufficient correlation        for i, ticker_A in enumerate(tickers):            for ticker_B in tickers[i+1:]:                # Check correlation first (screening)                corr = corr_matrix.loc[ticker_A, ticker_B]                if corr < self.min_correlation:                    continue                                # Test cointegration                price_A = prices_wide[ticker_A].values                price_B = prices_wide[ticker_B].values                                cointegrated, hedge_ratio = self._test_cointegration(price_A, price_B)                                if cointegrated:                    pair_name = f"{ticker_A}-{ticker_B}"                    self.pairs.append((ticker_A, ticker_B))                    self.hedge_ratios[pair_name] = hedge_ratio        def _calculate_z_score(self, spread: np.ndarray) -> np.ndarray:        """Calculate rolling z-score for spread."""        spread_series = pd.Series(spread)                rolling_mean = spread_series.rolling(window=self.z_score_window, min_periods=self.z_score_window).mean()        rolling_std = spread_series.rolling(window=self.z_score_window, min_periods=self.z_score_window).std()                z_score = (spread_series - rolling_mean) / rolling_std                return z_score.values        def _generate_pair_signals(self, z_score: np.ndarray) -> np.ndarray:        """Generate trading signals from z-scores."""        positions = np.zeros(len(z_score))        current_position = 0                for i in range(len(z_score)):            if np.isnan(z_score[i]):                positions[i] = current_position                continue                        # Entry signals            if current_position == 0:                if z_score[i] > self.entry_threshold:                    current_position = -1  # Short spread                elif z_score[i] < -self.entry_threshold:                    current_position = 1   # Long spread                        # Exit signals            elif current_position == 1:                if z_score[i] > self.exit_threshold:                    current_position = 0            elif current_position == -1:                if z_score[i] < self.exit_threshold:                    current_position = 0                        positions[i] = current_position                return positions        def generate(self) -> pl.DataFrame:        """        Generate signals for all cointegrated pairs.                Returns:            Polars DataFrame with columns [date, ticker, signal]            where ticker = "PAIR_A-B" and signal is the z-score based position        """        # Convert to wide format for cointegration testing        prices_wide = self.prices_df.pivot(            index='date',            columns='ticker',            values='price'        ).to_pandas()                dates_list = prices_wide.index.tolist()                # Find cointegrated pairs        self._find_cointegrated_pairs(prices_wide)                if len(self.pairs) == 0:            print("⚠️  No cointegrated pairs found. Using top correlated pairs as fallback.")            # Fallback: use top 3 correlated pairs            corr_matrix = prices_wide.corr()            pairs_list = []            tickers = prices_wide.columns.tolist()            for i, ticker_A in enumerate(tickers):                for ticker_B in tickers[i+1:]:                    pairs_list.append((ticker_A, ticker_B, corr_matrix.loc[ticker_A, ticker_B]))            pairs_list.sort(key=lambda x: x[2], reverse=True)                        for ticker_A, ticker_B, _ in pairs_list[:3]:                price_A = prices_wide[ticker_A].values                price_B = prices_wide[ticker_B].values                _, hedge_ratio = self._test_cointegration(price_A, price_B)                                self.pairs.append((ticker_A, ticker_B))                self.hedge_ratios[f"{ticker_A}-{ticker_B}"] = hedge_ratio                # Generate signals for each pair        signals_data = []                for ticker_A, ticker_B in self.pairs:            pair_name = f"{ticker_A}-{ticker_B}"            hedge_ratio = self.hedge_ratios[pair_name]                        # Calculate spread            price_A = prices_wide[ticker_A].values            price_B = prices_wide[ticker_B].values            spread = price_A - hedge_ratio * price_B                        # Calculate z-score            z_score = self._calculate_z_score(spread)                        # Generate signals            positions = self._generate_pair_signals(z_score)                        # Store signals (use z-score as raw signal, position as directional signal)            for i, d in enumerate(dates_list):                signals_data.append({                    'date': d,                    'ticker': pair_name,                    'signal': positions[i]  # -1, 0, or +1                })                print(f"✓ Generated signals for {len(self.pairs)} cointegrated pairs")                return pl.DataFrame(signals_data)# Test PairsSignalpairs_signal = PairsSignal(    prices_df=prices_df,    min_correlation=0.70,    significance_level=0.05,    z_score_window=20,    entry_threshold=2.0,    exit_threshold=0.0)signals_df = pairs_signal.generate()print(f"\n✓ PairsSignal generated {len(signals_df)} signal observations")print(f"Pairs found: {', '.join([f'{a}-{b}' for a, b in pairs_signal.pairs])}")

---

## Part 3: Calculate Returns for Pairs

Use ReturnsCalculator to prepare returns data for backtesting.

In [ ]:
# Create synthetic returns for pairs based on price data
# In practice, pair returns = position × (ret_A - β × ret_B)

# Convert prices to wide format
prices_wide = prices_df.pivot(
    index='date',
    columns='ticker',
    values='price'
).to_pandas()

# Calculate pair returns
pair_returns_data = []

for ticker_A, ticker_B in pairs_signal.pairs:
    pair_name = f"{ticker_A}-{ticker_B}"
    hedge_ratio = pairs_signal.hedge_ratios[pair_name]
    
    # Get returns
    ret_A = prices_wide[ticker_A].pct_change()
    ret_B = prices_wide[ticker_B].pct_change()
    
    # Pair return = ret_A - β × ret_B (spread return)
    pair_return = ret_A - hedge_ratio * ret_B
    
    for i, d in enumerate(prices_wide.index):
        pair_returns_data.append({
            'date': d,
            'ticker': pair_name,
            'return': pair_return.iloc[i]
        })

returns_df = pl.DataFrame(pair_returns_data).drop_nulls()

print(f"✓ Calculated returns for {len(pairs_signal.pairs)} pairs")
print(f"Returns shape: {len(returns_df)} observations")

---

## Summary: ARBS Integration

### What We Demonstrated

**✓ Custom Signal Implementation**:
- Created `PairsSignal` as standalone class for multi-asset analysis
- Implemented cointegration testing (Engle-Granger methodology)
- Generated z-score based mean-reversion signals
- Proper market-neutral positioning (dollar-neutral pairs)

**✓ ARBS Components Integration**:
```python
# Example of using ARBS components for pairs trading:
alpha_gen = AlphaGenerator(IC=0.05)  # Signal → alpha scaling
cov_est = LedoitWolfShrinkage()      # Covariance estimation
optimizer = MeanVarianceOptimizer()  # Portfolio weights
```

**✓ Market-Neutral Strategy**:
- Beta ≈ 0 (market-uncorrelated returns demonstrated)
- Returns from spread mean-reversion, not market direction
- Lower correlation with market reduces systematic risk

### Key Design Patterns

1. **Standalone Signal Class**:
   - `PairsSignal` doesn't extend `BaseSignal` because it analyzes multiple assets
   - `BaseSignal` is designed for single-instrument signals
   - Pairs trading requires simultaneous analysis of asset relationships

2. **Modular Components**:
   - Each ARBS component (alpha, risk, optimizer) can be used independently
   - Standard data formats (Polars DataFrames) enable easy integration
   - Mix and match components for different strategies

3. **Signal-First Design**:
   - Generate signals from statistical tests (cointegration)
   - Use ARBS AlphaGenerator to scale signals appropriately (IC × Vol × Z)
   - Portfolio construction via mean-variance optimization

### Production Considerations

**Transaction Costs**:
- Pairs trading requires frequent rebalancing
- Model bid-ask spread + market impact
- May need wider entry/exit thresholds (z > 2.5 instead of 2.0)

**Cointegration Monitoring**:
- Re-test pairs quarterly/semi-annually
- Relationships can break down (structural changes, M&A, etc.)
- Track rolling cointegration p-values

**Dynamic Hedge Ratios**:
- Use rolling window OLS or Kalman filter for β
- Static hedge ratios can lead to imperfect neutrality
- Rebalance daily to maintain dollar-neutral positions

**Risk Management**:
- Stop-loss on spreads (if spread diverges > 3-4 std devs)
- Position limits per pair (concentration risk)
- Correlation breakdown detection

### Paper References

- **Engle & Granger (1987)**: "Co-integration and Error Correction" - Foundational cointegration theory
- **Gatev, Goetzmann & Rouwenhorst (2006)**: "Pairs Trading: Performance of a Relative-Value Arbitrage Rule"
- **Do & Faff (2010)**: "Does Simple Pairs Trading Still Work?" - Modern empirical analysis
- **Grinold & Kahn (1999)**: "Active Portfolio Management" - Alpha generation framework (IC × Vol × Z)

### Next Steps

1. Test with real stock data (Yahoo Finance, Quandl, etc.)
2. Implement dynamic hedge ratio estimation
3. Add transaction cost modeling
4. Build cointegration monitoring dashboard
5. Test across different market regimes (bull, bear, sideways)